# v25 — Batasan ADX Atas (Ceiling) & Exhaustion Berbasis ADX Tinggi

**Latar belakang:** Investigasi 3 loss beruntun terakhir di trade log live (21 Agustus 2026,
diverifikasi ulang langsung dari candle MT5 HFM) menemukan pola konsisten: entry BUY terjadi
TEPAT di puncak momentum lokal (ADX 35-54, `bull_chain` 7-8 -- "hampir/sudah penuh"), lalu harga
langsung koreksi/berbalik 1-2 candle kemudian. Ini BUKAN soal ranging (ADX tetap >25 di semua
kasus) -- ini soal ADX/momentum yang SUDAH TERLALU TINGGI/matang, rawan exhaustion.

v13 sudah py exhaustion handling (SL/TP diperkecil 1.25x/1.0x ATR) TAPI cuma dipicu dari
`bull_chain`/`bear_chain` >= 8 (skala momentum chain), BUKAN dari ADX langsung. Kasus2 loss yg
diinvestigasi py chain 7 (blm masuk exhaustion mode) atau ADX tinggi dgn chain moderate (5-6)
-- celah yg belum tertangani.

**Ide user, 3 eksperimen (SEMUA dicoba & dibandingkan)**:
1. **ADX ceiling (cap atas)**: skip entry kalau ADX > threshold tertentu (mis. 45/50/55) --
   dianggap trend sudah terlalu matang, simetris dgn `adx_min` yg sudah ada (skip dari BAWAH).
2. **ADX-triggered exhaustion mode**: kalau ADX > threshold, JANGAN skip, tapi pakai SL/TP
   diperkecil (mirip exhaustion mode chain>=8 yg sudah ada, tapi dipicu ADX bukan chain).
3. **Geser `adx_min` (batas bawah) lebih rendah dari 18**: coba 12/15/18, dikombinasikan dgn
   eksperimen 1 & 2 di atas -- cari rentang ADX (min-max) yang optimal scr menyeluruh.

**Metodologi**: sama persis pola v19-v24 -- TRAIN (2019-2023) / TEST (2024-2026) split, kriteria
kejujuran PF>1.5 required + sample TRAIN>=30/TEST>=15, spread real 1.82, v12_score ASLI +
Order Block filter + H1 alignment (BUKAN reimplementasi). Reuse cache v23
(`df_2019_2026_full_mtf.parquet`).

**TIDAK ADA perubahan ke `usecase.py`** -- murni riset backtest.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STRATEGY_NAME = "m5_scalping"
VERSION = "v25"

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME
EXPORT_DIR = PROJECT_ROOT / "dataset" / "exports" / STRATEGY_NAME / VERSION
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

INITIAL_EQUITY = 100.0
RISK_PCT = 0.01
CONTRACT_SIZE = 100.0
MIN_LOT = 0.01
LOT_STEP = 0.01
REAL_SPREAD = 1.82

MIN_SAMPLE_TRAIN = 30
MIN_SAMPLE_TEST = 15

pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Load cache v23 (skor v12 + OB + H1 EMA, 2019-2026)

In [2]:
v23_cache = PROCESSED_DIR / "v23" / "df_2019_2026_full_mtf.parquet"
assert v23_cache.exists(), "Cache v23 belum ada -- perlu dijalankan dulu"
df = pd.read_parquet(v23_cache)
print(f"Load dari cache v23: {len(df)} candle, {df['datetime'].min()} -> {df['datetime'].max()}")
print(f"Kolom: {list(df.columns)}")

print(f"\nDistribusi ADX (semua candle):")
print(df['adx'].describe())

Load dari cache v23: 518403 candle, 2019-01-01 23:00:00+00:00 -> 2026-08-06 12:35:00+00:00
Kolom: ['datetime', 'open', 'high', 'low', 'close', 'adx', 'atr', 'v12_score', 'bull_chain', 'bear_chain', 'bos_bull', 'bos_bear', 'ob_bull', 'ob_bear', 'h1_ob_bull', 'h1_ob_bear', 'h1_ema_50', 'h1_ema_200', 'h1_bos_bull', 'h1_bos_bear']

Distribusi ADX (semua candle):
count    518403.000000
mean         24.826185
std          10.501662
min           0.000000
25%          17.027933
50%          22.572038
75%          30.449570
max          83.850744
Name: adx, dtype: float64


## 2. Backtest engine v25: adx_min + adx_ceiling (skip) + adx_exhaustion (SL/TP kecil)

Urutan logika per candle (adx_min <= ADX): (a) kalau `adx_ceiling` aktif DAN ADX > ceiling ->
SKIP entry sama sekali; (b) kalau tidak, cek apakah ADX > `adx_exhaustion_threshold` -> pakai
SL/TP mode exhaustion (lebih kecil); (c) selain itu -> SL/TP mode normal. `adx_ceiling` dan
`adx_exhaustion_threshold` adalah 2 mekanisme TERPISAH yg bisa dipakai sendiri2 atau kombinasi
(kalau exhaustion_threshold < ceiling, exhaustion jalan dulu sblm ceiling skip total).

In [3]:
def check_h1_alignment_v25(h1_ema_50, h1_ema_200, direction: str) -> bool:
    if h1_ema_50 is None or h1_ema_200 is None or not np.isfinite(h1_ema_50) or not np.isfinite(h1_ema_200):
        return True
    h1_trend = "UP" if h1_ema_50 > h1_ema_200 else ("DOWN" if h1_ema_50 < h1_ema_200 else "FLAT")
    if direction == "BUY" and h1_trend == "DOWN":
        return False
    if direction == "SELL" and h1_trend == "UP":
        return False
    return True


def run_backtest_v25(
    df_signals: pd.DataFrame,
    adx_min: float = 18.0,
    adx_ceiling: float = None,             # None = tanpa cap (skip mekanisme A)
    adx_exhaustion_threshold: float = None, # None = tanpa exhaustion-by-ADX (skip mekanisme B)
    min_signal_score: float = 9.0,
    sl_mult_normal: float = 2.0,
    tp_mult_normal: float = 4.0,
    sl_mult_exhaustion: float = 1.25,
    tp_mult_exhaustion: float = 1.0,
    max_hold_normal: int = 12,
    max_hold_exhaustion: int = 6,
    require_ob_filter: bool = True,
    require_h1_alignment: bool = True,
    spread_points: float = REAL_SPREAD,
) -> pd.DataFrame:
    close_arr = df_signals["close"].to_numpy()
    high_arr = df_signals["high"].to_numpy()
    low_arr = df_signals["low"].to_numpy()
    adx_arr = df_signals["adx"].to_numpy()
    atr_arr = df_signals["atr"].to_numpy()
    score_arr = df_signals["v12_score"].to_numpy()
    ob_bull_arr = df_signals["ob_bull"].to_numpy()
    ob_bear_arr = df_signals["ob_bear"].to_numpy()
    h1_ob_bull_arr = df_signals["h1_ob_bull"].to_numpy()
    h1_ob_bear_arr = df_signals["h1_ob_bear"].to_numpy()
    h1_ema_50_arr = df_signals["h1_ema_50"].to_numpy()
    h1_ema_200_arr = df_signals["h1_ema_200"].to_numpy()
    datetime_arr = df_signals["datetime"].to_numpy()
    n = len(df_signals)

    trades = []
    equity = INITIAL_EQUITY
    i = 0
    while i < n:
        adx, atr, close, score = adx_arr[i], atr_arr[i], close_arr[i], score_arr[i]
        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(adx) or not np.isfinite(score):
            i += 1
            continue

        if adx < adx_min:
            i += 1
            continue
        if adx_ceiling is not None and adx > adx_ceiling:
            i += 1
            continue

        direction = None
        if score >= min_signal_score:
            direction = "BUY"
        elif score <= -min_signal_score:
            direction = "SELL"
        if direction is None:
            i += 1
            continue

        if require_ob_filter:
            opposing_ob = (
                (direction == "BUY" and (ob_bear_arr[i] > 0 or h1_ob_bear_arr[i] > 0)) or
                (direction == "SELL" and (ob_bull_arr[i] > 0 or h1_ob_bull_arr[i] > 0))
            )
            if opposing_ob:
                i += 1
                continue
        if require_h1_alignment:
            if not check_h1_alignment_v25(h1_ema_50_arr[i], h1_ema_200_arr[i], direction):
                i += 1
                continue

        is_exhausted = adx_exhaustion_threshold is not None and adx > adx_exhaustion_threshold
        if is_exhausted:
            sl_mult, tp_mult, max_hold, mode = sl_mult_exhaustion, tp_mult_exhaustion, max_hold_exhaustion, "EXHAUSTION"
        else:
            sl_mult, tp_mult, max_hold, mode = sl_mult_normal, tp_mult_normal, max_hold_normal, "NORMAL"

        sl_points = sl_mult * atr
        tp_points = tp_mult * atr
        entry_price = close + (spread_points if direction == "BUY" else -spread_points)
        tp_price = entry_price + tp_points if direction == "BUY" else entry_price - tp_points
        sl_price = entry_price - sl_points if direction == "BUY" else entry_price + sl_points

        entry_time = datetime_arr[i]
        exit_price = None
        exit_idx = min(i + max_hold, n - 1)
        window_end = min(i + 1 + max_hold, n)
        for candle_idx in range(i + 1, window_end):
            c_high, c_low = high_arr[candle_idx], low_arr[candle_idx]
            hit_tp = c_high >= tp_price if direction == "BUY" else c_low <= tp_price
            hit_sl = c_low <= sl_price if direction == "BUY" else c_high >= sl_price
            if hit_sl:
                exit_price, exit_time = sl_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_time = tp_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
        if exit_price is None:
            exit_price, exit_time = close_arr[exit_idx], datetime_arr[exit_idx]

        next_i = exit_idx + 1
        price_move = (exit_price - entry_price) if direction == "BUY" else (entry_price - exit_price)
        risk_amount = equity * RISK_PCT
        lot = max(round(math.floor((risk_amount / (sl_points * CONTRACT_SIZE)) / LOT_STEP) * LOT_STEP, 2), MIN_LOT) if sl_points > 0 else MIN_LOT
        pnl = price_move * lot * CONTRACT_SIZE
        equity += pnl
        trades.append({
            "entry_time": entry_time, "mode": mode, "direction": direction, "adx_at_entry": adx, "pnl": pnl,
            "result": "WIN" if pnl > 0 else "LOSS", "equity_after": equity,
        })
        i = next_i

    return pd.DataFrame(trades)


def evaluate(trades: pd.DataFrame, initial_equity: float) -> dict:
    if trades.empty:
        return {"total_trades": 0, "win_rate_pct": 0, "profit_factor": 0, "net_pnl": 0, "max_drawdown_pct": 0}
    wins = trades[trades["pnl"] > 0]
    losses = trades[trades["pnl"] <= 0]
    gross_profit = wins["pnl"].sum()
    gross_loss = losses["pnl"].sum()
    equity_series = pd.Series([initial_equity] + trades["equity_after"].tolist())
    running_max = equity_series.cummax()
    drawdown = (equity_series - running_max) / running_max * 100
    return {
        "total_trades": len(trades),
        "win_rate_pct": round(len(wins) / len(trades) * 100, 2),
        "profit_factor": round(gross_profit / abs(gross_loss), 2) if gross_loss != 0 else float("inf"),
        "net_pnl": round(gross_profit + gross_loss, 2),
        "max_drawdown_pct": round(drawdown.min(), 2),
    }

print("Backtest engine v25 siap.")

Backtest engine v25 siap.


## 3. TRAIN/TEST split & Baseline (v13 murni, adx_min=18, tanpa ceiling/exhaustion-ADX)

In [4]:
TRAIN_END = pd.Timestamp("2024-01-01", tz="UTC")
df_train = df[df["datetime"] < TRAIN_END].reset_index(drop=True)
df_test = df[df["datetime"] >= TRAIN_END].reset_index(drop=True)
print(f"TRAIN (2019-2023): {len(df_train)} candle | TEST (2024-2026): {len(df_test)} candle")

trades_base_train = run_backtest_v25(df_train, adx_min=18.0, adx_ceiling=None, adx_exhaustion_threshold=None)
trades_base_test = run_backtest_v25(df_test, adx_min=18.0, adx_ceiling=None, adx_exhaustion_threshold=None)
print("=== Baseline: v13 murni (adx_min=18, tanpa ceiling/exhaustion-ADX) ===")
print("TRAIN:", evaluate(trades_base_train, INITIAL_EQUITY))
print("TEST :", evaluate(trades_base_test, INITIAL_EQUITY))

TRAIN (2019-2023): 351136 candle | TEST (2024-2026): 167267 candle


=== Baseline: v13 murni (adx_min=18, tanpa ceiling/exhaustion-ADX) ===
TRAIN: {'total_trades': 2282, 'win_rate_pct': 15.69, 'profit_factor': np.float64(0.38), 'net_pnl': np.float64(-2348.28), 'max_drawdown_pct': np.float64(-2348.28)}
TEST : {'total_trades': 1105, 'win_rate_pct': 45.7, 'profit_factor': np.float64(1.58), 'net_pnl': np.float64(1588.28), 'max_drawdown_pct': np.float64(-229.55)}


## 4. Cek pola dulu: apakah win rate memang menurun seiring ADX makin tinggi? (baca data sblm grid search)

Sebelum grid search parameter, verifikasi dulu hipotesis dasarnya dari data TRAIN: apakah
trade dgn ADX sangat tinggi memang lebih sering rugi drpd ADX sedang -- kalau tidak ada pola
ini scr statistik, ceiling/exhaustion-ADX kemungkinan tidak akan membantu.

In [5]:
trades_base_train["adx_bucket"] = pd.cut(
    trades_base_train["adx_at_entry"],
    bins=[18, 25, 30, 35, 40, 45, 50, 100],
    labels=["18-25", "25-30", "30-35", "35-40", "40-45", "45-50", "50+"],
)
print("=== TRAIN: win rate & PF per bucket ADX saat entry ===")
bucket_stats = trades_base_train.groupby("adx_bucket", observed=True).apply(
    lambda g: pd.Series({
        "n": len(g),
        "win_rate_pct": round((g["result"]=="WIN").mean()*100, 1),
        "pf": round(g.loc[g.pnl>0,"pnl"].sum() / abs(g.loc[g.pnl<=0,"pnl"].sum()), 2) if (g.pnl<=0).any() else float("inf"),
        "avg_pnl": round(g["pnl"].mean(), 2),
    }), include_groups=False
)
print(bucket_stats.to_string())

=== TRAIN: win rate & PF per bucket ADX saat entry ===
                n  win_rate_pct    pf  avg_pnl
adx_bucket                                    
18-25       941.0          13.7  0.33    -1.11
25-30       497.0          16.3  0.37    -1.04
30-35       305.0          15.4  0.37    -1.01
35-40       218.0          18.3  0.58    -0.67
40-45       137.0          15.3  0.40    -0.99
45-50        91.0          25.3  0.64    -0.58
50+          93.0          18.3  0.29    -1.51


## 5. Grid search: adx_min (lebih rendah) x adx_ceiling x adx_exhaustion_threshold x SL/TP exhaustion

In [6]:
import itertools
import time as _time

GRID = {
    "adx_min": [12.0, 15.0, 18.0],
    "adx_ceiling": [None, 45.0, 50.0, 55.0, 60.0],
    "adx_exhaustion_threshold": [None, 35.0, 40.0, 45.0],
    "sl_mult_exhaustion": [1.0, 1.25, 1.5],
    "tp_mult_exhaustion": [1.0, 1.5, 2.0],
}
FIXED = dict(sl_mult_normal=2.0, tp_mult_normal=4.0, max_hold_normal=12, max_hold_exhaustion=6)

combos = list(itertools.product(*GRID.values()))
print(f"Total kombinasi grid (sblm filter): {len(combos)}")

t0 = _time.time()
grid_results = []
for idx, combo in enumerate(combos):
    params = dict(zip(GRID.keys(), combo))
    # Skip kombinasi redundan: kalau adx_exhaustion_threshold None, sl/tp_mult_exhaustion gak dipakai -- skip duplikat
    if params["adx_exhaustion_threshold"] is None and (params["sl_mult_exhaustion"] != 1.25 or params["tp_mult_exhaustion"] != 1.5):
        continue
    trades = run_backtest_v25(df_train, **params, **FIXED)
    metrics = evaluate(trades, INITIAL_EQUITY)
    metrics.update(params)
    grid_results.append(metrics)
    if (idx + 1) % 200 == 0:
        print(f"  [{idx+1}/{len(combos)}] {_time.time()-t0:.0f}s")

grid_df = pd.DataFrame(grid_results)
print(f"\nGrid search selesai dalam {_time.time()-t0:.0f}s ({len(grid_df)} kombinasi valid dievaluasi)")

grid_valid = grid_df[grid_df["total_trades"] >= MIN_SAMPLE_TRAIN].sort_values("profit_factor", ascending=False)
print(f"\n=== Top 20 kandidat (sample TRAIN >= {MIN_SAMPLE_TRAIN}) ===")
print(grid_valid.head(20).to_string(index=False))

baseline_pf_train = evaluate(trades_base_train, INITIAL_EQUITY)["profit_factor"]
print(f"\nBaseline v13 murni TRAIN PF: {baseline_pf_train}")
print(f"Kandidat yang MENGUNGGULI baseline (PF lebih tinggi): {(grid_valid['profit_factor'] > baseline_pf_train).sum()} dari {len(grid_valid)}")

Total kombinasi grid (sblm filter): 540


  [200/540] 136s



Grid search selesai dalam 370s (420 kombinasi valid dievaluasi)

=== Top 20 kandidat (sample TRAIN >= 30) ===
 total_trades  win_rate_pct  profit_factor  net_pnl  max_drawdown_pct  adx_min  adx_ceiling  adx_exhaustion_threshold  sl_mult_exhaustion  tp_mult_exhaustion
         2190         15.57           0.39 -2210.39          -2210.39     18.0         50.0                       NaN                1.25                 1.5
         2331         15.66           0.39 -2346.80          -2346.80     15.0         50.0                       NaN                1.25                 1.5
         2260         15.71           0.39 -2291.50          -2291.50     18.0         60.0                       NaN                1.25                 1.5
         2275         15.34           0.38 -2324.72          -2324.72     18.0         60.0                      45.0                1.50                 2.0
         2237         15.47           0.38 -2293.43          -2293.43     18.0         55.0        

## 6. Validasi TEST out-of-sample (kandidat yang mengungguli baseline TRAIN)

In [7]:
candidates_passing = grid_valid[grid_valid["profit_factor"] > baseline_pf_train].head(20)
print(f"Kandidat TRAIN mengungguli baseline: {len(candidates_passing)}")

if len(candidates_passing) == 0:
    print("\n>>> TIDAK ADA kandidat mengungguli baseline di TRAIN. Validasi TEST DIBATALKAN.")
else:
    test_results = []
    for _, row in candidates_passing.iterrows():
        params = {k: row[k] for k in GRID.keys()}
        trades_test = run_backtest_v25(df_test, **params, **FIXED)
        m_test = evaluate(trades_test, INITIAL_EQUITY)
        test_results.append({**params, "train_pf": row["profit_factor"], "train_n": row["total_trades"],
                              "test_pf": m_test["profit_factor"], "test_n": m_test["total_trades"],
                              "test_wr": m_test["win_rate_pct"], "test_netpnl": m_test["net_pnl"],
                              "test_maxdd": m_test["max_drawdown_pct"]})

    test_df = pd.DataFrame(test_results)
    print("\n=== Validasi TEST utk kandidat yang menang di TRAIN ===")
    print(test_df.to_string(index=False))

    baseline_test_metrics = evaluate(trades_base_test, INITIAL_EQUITY)
    print(f"\nBaseline v13 murni TEST: PF={baseline_test_metrics['profit_factor']}, net_pnl={baseline_test_metrics['net_pnl']}, "
          f"max_dd={baseline_test_metrics['max_drawdown_pct']}")

    robust = test_df[(test_df["test_pf"] > baseline_test_metrics["profit_factor"]) & (test_df["test_n"] >= MIN_SAMPLE_TEST)]
    print(f"\n>>> Kandidat ROBUST (unggul TRAIN & TEST vs baseline, sample TEST>={MIN_SAMPLE_TEST}): {len(robust)}")
    if len(robust) > 0:
        print(robust.to_string(index=False))

Kandidat TRAIN mengungguli baseline: 3



=== Validasi TEST utk kandidat yang menang di TRAIN ===
 adx_min  adx_ceiling  adx_exhaustion_threshold  sl_mult_exhaustion  tp_mult_exhaustion  train_pf  train_n  test_pf  test_n  test_wr  test_netpnl  test_maxdd
    18.0         50.0                       NaN                1.25                 1.5      0.39   2190.0     1.52    1053    45.20      1324.87     -236.52
    15.0         50.0                       NaN                1.25                 1.5      0.39   2331.0     1.55    1123    45.33      1460.02     -234.51
    18.0         60.0                       NaN                1.25                 1.5      0.39   2260.0     1.55    1090    45.41      1475.72     -237.05

Baseline v13 murni TEST: PF=1.58, net_pnl=1588.28, max_dd=-229.55

>>> Kandidat ROBUST (unggul TRAIN & TEST vs baseline, sample TEST>=15): 0


## 7. Kesimpulan

*(diisi setelah lihat hasil eksekusi lengkap Section 3-6 -- placeholder)*